# Capítulo 1: Estrutura Básica de Prompts

- [Lição](#lesson)
- [Exercícios](#exercises)
- [Área de Testes](#example-playground)

## Configuração

Execute a célula de configuração abaixo para carregar sua chave de API e estabelecer a função auxiliar `get_completion`.

In [ ]:
!pip install anthropic

# Import python's built-in regular expression library
import re
import anthropic

# Retrieve the API_KEY & MODEL_NAME variables from the IPython store
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## Lição

A Anthropic oferece duas APIs, a legada [Text Completions API](https://docs.anthropic.com/claude/reference/complete_post) e a atual [Messages API](https://docs.anthropic.com/claude/reference/messages_post). Para este tutorial, usaremos exclusivamente a Messages API.

No mínimo, uma chamada ao Claude usando a Messages API requer os seguintes parâmetros:
- `model`: o [nome do modelo na API](https://docs.anthropic.com/claude/docs/models-overview#model-recommendations) do modelo que você pretende chamar

- `max_tokens`: o número máximo de tokens a gerar antes de parar. Note que o Claude pode parar antes de atingir este máximo. Este parâmetro especifica apenas o número máximo absoluto de tokens a gerar. Além disso, esta é uma parada *hard*, o que significa que pode fazer com que o Claude pare de gerar no meio de uma palavra ou frase.

- `messages`: um array de mensagens de entrada. Nossos modelos são treinados para operar em turnos conversacionais alternados entre `user` e `assistant`. Ao criar uma nova `Message`, você especifica os turnos conversacionais anteriores com o parâmetro messages, e o modelo então gera a próxima `Message` na conversa.
  - Cada mensagem de entrada deve ser um objeto com `role` e `content`. Você pode especificar uma única mensagem com role `user`, ou pode incluir múltiplas mensagens `user` e `assistant` (elas devem alternar, nesse caso). A primeira mensagem deve sempre usar o role `user`.

Também existem parâmetros opcionais, tais como:
- `system`: o system prompt - mais sobre isso abaixo.
  
- `temperature`: o grau de variabilidade na resposta do Claude. Para estas lições e exercícios, definimos `temperature` como 0.

Para uma lista completa de todos os parâmetros da API, visite nossa [documentação da API](https://docs.anthropic.com/claude/reference/messages_post).

### Exemplos

Vamos ver como o Claude responde a alguns prompts formatados corretamente. Para cada uma das células a seguir, execute a célula (`shift+enter`), e a resposta do Claude aparecerá abaixo do bloco.

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

Agora vamos ver alguns prompts que não incluem a formatação correta da Messages API. Para esses prompts mal formatados, a Messages API retorna um erro.

Primeiro, temos um exemplo de uma chamada da Messages API que não possui os campos `role` e `content` no array `messages`.

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

Aqui está um prompt que falha ao alternar entre os roles `user` e `assistant`.

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

Mensagens `user` e `assistant` **DEVEM alternar**, e as mensagens **DEVEM começar com um turno `user`**. Você pode ter múltiplos pares `user` & `assistant` em um prompt (como se estivesse simulando uma conversa de múltiplos turnos). Você também pode colocar palavras em uma mensagem terminal `assistant` para que o Claude continue de onde você parou (mais sobre isso em capítulos posteriores).

#### System Prompts

Você também pode usar **system prompts**. Um system prompt é uma forma de **fornecer contexto, instruções e diretrizes ao Claude** antes de apresentar uma pergunta ou tarefa no turno "User".

Estruturalmente, system prompts existem separadamente da lista de mensagens `user` & `assistant`, e portanto pertencem a um parâmetro `system` separado (dê uma olhada na estrutura da função auxiliar `get_completion` na seção [Configuração](#setup) do notebook).

Neste tutorial, onde quer que possamos utilizar um system prompt, fornecemos um campo `system` na sua função de completions. Se você não quiser usar um system prompt, simplesmente defina a variável `SYSTEM_PROMPT` como uma string vazia.

#### Exemplo de System Prompt

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))

Por que usar um system prompt? Um **system prompt bem escrito pode melhorar o desempenho do Claude** de várias formas, como aumentar a capacidade do Claude de seguir regras e instruções. Para mais informações, visite nossa documentação sobre [como usar system prompts](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) com o Claude.

Agora vamos mergulhar em alguns exercícios. Se você quiser experimentar com os prompts da lição sem alterar nenhum conteúdo acima, role até o final do notebook da lição para visitar a [**Área de Testes**](#example-playground).

---

## Exercícios
- [Exercício 1.1 - Contando até Três](#exercise-11---counting-to-three)
- [Exercício 1.2 - System Prompt](#exercise-12---system-prompt)

### Exercício 1.1 - Contando até Três
Usando a formatação adequada de `user` / `assistant`, edite o `PROMPT` abaixo para fazer o Claude **contar até três**. A saída também indicará se sua solução está correta.

In [ ]:
# Prompt - this is the only field you should change
PROMPT = "[Replace this text]"

# Get Claude's response
response = get_completion(PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_1_1_hint; print(exercise_1_1_hint)

### Exercício 1.2 - System Prompt

Modifique o `SYSTEM_PROMPT` para fazer o Claude responder como se fosse uma criança de 3 anos.

In [ ]:
# System prompt - this is the only field you should change
SYSTEM_PROMPT = "[Replace this text]"

# Prompt
PROMPT = "How big is the sky?"

# Get Claude's response
response = get_completion(PROMPT, SYSTEM_PROMPT)

# Function to grade exercise correctness
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# Print Claude's response and the corresponding grade
print(response)
print("\n--------------------------- GRADING ---------------------------")
print("This exercise has been correctly solved:", grade_exercise(response))

❓ Se você quiser uma dica, execute a célula abaixo!

In [ ]:
from hints import exercise_1_2_hint; print(exercise_1_2_hint)

### Parabéns!

Se você resolveu todos os exercícios até este ponto, está pronto para avançar para o próximo capítulo. Bons prompts!

---

## Área de Testes

Esta é uma área para você experimentar livremente com os exemplos de prompt mostrados nesta lição e ajustar os prompts para ver como isso pode afetar as respostas do Claude.

In [ ]:
# Prompt
PROMPT = "Hi Claude, how are you?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "Can you tell me the color of the ocean?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Prompt
PROMPT = "What year was Celine Dion born in?"

# Print Claude's response
print(get_completion(PROMPT))

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# Get Claude's response
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# Print Claude's response
print(response[0].text)

In [ ]:
# System prompt
SYSTEM_PROMPT = "Your answer should always be a series of critical thinking questions that further the conversation (do not provide answers to your questions). Do not actually answer the user question."

# Prompt
PROMPT = "Why is the sky blue?"

# Print Claude's response
print(get_completion(PROMPT, SYSTEM_PROMPT))